In [ ]:
%load_ext d2lbook.tab
tab.interact_select(['mxnet', 'pytorch', 'tensorflow', 'jax'])




# Conception d'architectures de réseaux de neurones convolutifs
:label:`sec_cnn-design`

Les sections précédentes nous ont fait découvrir la conception des réseaux modernes pour la vision par ordinateur. Un point commun à tous les travaux que nous avons abordés est qu'ils reposaient largement sur l'intuition des chercheurs. De nombreuses architectures sont fortement influencées par la créativité humaine et, dans une bien moindre mesure, par une exploration systématique de l'espace de conception offert par les réseaux profonds. Néanmoins, cette approche d'*ingénierie de réseau* (network engineering) a connu un succès retentissant.

Depuis qu'AlexNet (:numref:`sec_alexnet`) a détrôné les modèles de vision par ordinateur conventionnels sur ImageNet, il est devenu courant de construire des réseaux très profonds en empilant des blocs de convolutions, tous conçus selon le même motif. En particulier, les convolutions $3 \times 3$ ont été popularisées par les réseaux VGG (:numref:`sec_vgg`). NiN (:numref:`sec_nin`) a montré que même les convolutions $1 \times 1$ pouvaient être bénéfiques en ajoutant des non-linéarités locales. De plus, NiN a résolu le problème de l'agrégation de l'information en tête de réseau en agrégeant sur toutes les positions. GoogLeNet (:numref:`sec_googlenet`) a ajouté plusieurs branches de différentes largeurs de convolution, combinant les avantages de VGG et NiN dans son bloc Inception. Les ResNets (:numref:`sec_resnet`) ont orienté le biais inductif vers la fonction identité (au lieu de $f(x) = 0$). Cela a permis d'obtenir des réseaux très profonds. Près d'une décennie plus tard, la conception ResNet est toujours populaire, ce qui témoigne de sa robustesse. Enfin, ResNeXt (:numref:`subsec_resnext`) a ajouté des convolutions groupées, offrant un meilleur compromis entre paramètres et calcul. Précurseurs des Transformers pour la vision, les réseaux Squeeze-and-Excitation (SENets) permettent un transfert d'information efficace entre les positions :cite:`Hu.Shen.Sun.2018`. Cela a été accompli en calculant une fonction d'attention globale par canal.

Jusqu'à présent, nous avons omis les réseaux obtenus via la *recherche d'architecture neuronale* (NAS pour *neural architecture search*) :cite:`zoph2016neural,liu2018darts`. Nous avons choisi de le faire car leur coût est généralement énorme, reposant sur la recherche par force brute, les algorithmes génétiques, l'apprentissage par renforcement ou toute autre forme d'optimisation d'hyperparamètres. Étant donné un espace de recherche fixe, la NAS utilise une stratégie de recherche pour sélectionner automatiquement une architecture basée sur l'estimation des performances obtenue. Le résultat de la NAS est une instance de réseau unique. Les EfficientNets sont un résultat notable de cette recherche :cite:`tan2019efficientnet`.

Dans ce qui suit, nous discutons d'une idée qui est assez différente de la quête du *meilleur réseau unique*. Elle est relativement peu coûteuse en termes de calcul, elle apporte des connaissances scientifiques en cours de route et elle est très efficace en termes de qualité des résultats. Passons en revue la stratégie de :citet:`Radosavovic.Kosaraju.Girshick.ea.2020` pour *concevoir des espaces de conception de réseaux*. La stratégie combine la force de la conception manuelle et de la NAS. Elle y parvient en opérant sur des *distributions de réseaux* et en optimisant les distributions de manière à obtenir de bonnes performances pour des familles entières de réseaux. Le résultat de cette approche sont les *RegNets*, spécifiquement RegNetX et RegNetY, ainsi qu'une série de principes directeurs pour la conception de CNN performants.



In [ ]:
%%tab mxnet
from d2l import mxnet as d2l
from mxnet import np, npx, init
from mxnet.gluon import nn

npx.set_np()


In [ ]:
%%tab pytorch
from d2l import torch as d2l
import torch
from torch import nn
from torch.nn import functional as F


In [ ]:
%%tab tensorflow
import tensorflow as tf
from d2l import tensorflow as d2l


In [ ]:
%%tab jax
from d2l import jax as d2l
from flax import linen as nn




## L'espace de conception AnyNet
:label:`subsec_the-anynet-design-space`

La description ci-dessous suit de près le raisonnement de :citet:`Radosavovic.Kosaraju.Girshick.ea.2020` avec quelques abréviations pour l'adapter au cadre de ce livre.
Pour commencer, nous avons besoin d'un modèle pour la famille de réseaux à explorer. L'un des points communs des conceptions de ce chapitre est que les réseaux se composent d'une *tige* (stem), d'un *corps* (body) et d'une *tête* (head). La tige effectue le traitement initial de l'image, souvent par des convolutions avec une taille de fenêtre plus grande. Le corps se compose de plusieurs blocs, effectuant l'essentiel des transformations nécessaires pour passer des images brutes aux représentations d'objets. Enfin, la tête convertit cela en sorties souhaitées, par exemple via un régresseur softmax pour la classification multiclasse.
Le corps, à son tour, se compose de plusieurs étapes (stages), opérant sur l'image à des résolutions décroissantes. En fait, la tige et chaque étape suivante divisent par quatre la résolution spatiale. Enfin, chaque étape se compose d'un ou plusieurs blocs. Ce motif est commun à tous les réseaux, de VGG à ResNeXt. En effet, pour la conception de réseaux AnyNet génériques, :citet:`Radosavovic.Kosaraju.Girshick.ea.2020` ont utilisé le bloc ResNeXt de la :numref:`fig_resnext_block`.


![L'espace de conception AnyNet. Les nombres $(\mathit{c}, \mathit{r})$ le long de chaque flèche indiquent le nombre de canaux $c$ et la résolution $\mathit{r} \times \mathit{r}$ des images à ce point. De gauche à droite : structure générique du réseau composée d'une tige, d'un corps et d'une tête ; corps composé de quatre étapes ; structure détaillée d'une étape ; deux structures alternatives pour les blocs, l'une sans sous-échantillonnage et l'autre qui divise par deux la résolution dans chaque dimension. Les choix de conception incluent la profondeur $\mathit{d_i}$, le nombre de canaux de sortie $\mathit{c_i}$, le nombre de groupes $\mathit{g_i}$ et le rapport de goulot d'étranglement (bottleneck ratio) $\mathit{k_i}$ pour n'importe quelle étape $\mathit{i}$.](../img/anynet.svg)
:label:`fig_anynet_full`

Examinons en détail la structure décrite dans la :numref:`fig_anynet_full`. Comme mentionné, un AnyNet se compose d'une tige, d'un corps et d'une tête. La tige prend en entrée des images RVB (3 canaux), utilise une convolution $3 \times 3$ avec une foulée de $2$, suivie d'une normalisation par lots, pour diviser par deux la résolution de $r \times r$ à $r/2 \times r/2$. De plus, elle génère $c_0$ canaux qui servent d'entrée au corps. 

Puisque le réseau est conçu pour bien fonctionner avec les images ImageNet de forme $224 \times 224 \times 3$, le corps sert à réduire cela à $7 \times 7 \times c_4$ via 4 étapes (rappelons que $224 / 2^{1+4} = 7$), chacune avec une éventuelle foulée de $2$. Enfin, la tête utilise une conception tout à fait standard via un pooling moyen global, similaire à NiN (:numref:`sec_nin`), suivi d'une couche entièrement connectée pour émettre un vecteur de dimension $n$ pour une classification à $n$ classes. 

La plupart des décisions de conception pertinentes sont inhérentes au corps du réseau. Il procède par étapes, où chaque étape est composée du même type de blocs ResNeXt que ceux que nous avons vus dans la :numref:`subsec_resnext`. La conception y est à nouveau entièrement générique : nous commençons par un bloc qui divise par deux la résolution en utilisant une foulée de $2$ (le plus à droite dans la :numref:`fig_anynet_full`). Pour s'adapter à cela, la branche résiduelle du bloc ResNeXt doit passer par une convolution $1 \times 1$. Ce bloc est suivi d'un nombre variable de blocs ResNeXt supplémentaires qui laissent inchangés la résolution et le nombre de canaux. Notez qu'une pratique de conception courante consiste à ajouter un léger goulot d'étranglement (bottleneck) dans la conception des blocs convolutifs. 
En tant que tel, avec un rapport de goulot d'étranglement $k_i \geq 1$, nous allouons un certain nombre de canaux, $c_i/k_i$, au sein de chaque bloc pour l'étape $i$ (comme le montrent les expériences, ce n'est pas vraiment efficace et cela devrait être évité). Enfin, puisque nous avons affaire à des blocs ResNeXt, nous devons également choisir le nombre de groupes $g_i$ pour les convolutions groupées à l'étape $i$. 

Cet espace de conception apparemment générique nous offre néanmoins de nombreux paramètres : nous pouvons définir la largeur de bloc (nombre de canaux) $c_0, \ldots c_4$, la profondeur (nombre de blocs) par étape $d_1, \ldots d_4$, les rapports de goulot d'étranglement $k_1, \ldots k_4$ et les largeurs de groupe (nombres de groupes) $g_1, \ldots g_4$. 
Au total, cela représente 17 paramètres, ce qui donne un nombre déraisonnablement élevé de configurations qu'il conviendrait d'explorer. Nous avons besoin d'outils pour réduire efficacement cet immense espace de conception. C'est là qu'intervient la beauté conceptuelle des espaces de conception. Avant de le faire, implémentons d'abord la conception générique.



In [ ]:
%%tab mxnet
class AnyNet(d2l.Classifier):
    def stem(self, num_channels):
        net = nn.Sequential()
        net.add(nn.Conv2D(num_channels, kernel_size=3, padding=1, strides=2),
                nn.BatchNorm(), nn.Activation('relu'))
        return net


In [ ]:
%%tab pytorch
class AnyNet(d2l.Classifier):
    def stem(self, num_channels):
        return nn.Sequential(
            nn.LazyConv2d(num_channels, kernel_size=3, stride=2, padding=1),
            nn.LazyBatchNorm2d(), nn.ReLU())


In [ ]:
%%tab tensorflow
class AnyNet(d2l.Classifier):
    def stem(self, num_channels):
        return tf.keras.models.Sequential([
            tf.keras.layers.Conv2D(num_channels, kernel_size=3, strides=2,
                                   padding='same'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Activation('relu')])


In [ ]:
%%tab jax
class AnyNet(d2l.Classifier):
    arch: tuple
    stem_channels: int
    lr: float = 0.1
    num_classes: int = 10
    training: bool = True

    def setup(self):
        self.net = self.create_net()

    def stem(self, num_channels):
        return nn.Sequential([
            nn.Conv(num_channels, kernel_size=(3, 3), strides=(2, 2),
                    padding=(1, 1)),
            nn.BatchNorm(not self.training),
            nn.relu
        ])




Chaque étape se compose de `depth` blocs ResNeXt, où `num_channels` spécifie la largeur du bloc. Notez que le premier bloc divise par deux la hauteur et la largeur des images d'entrée.



In [ ]:
%%tab mxnet
@d2l.add_to_class(AnyNet)
def stage(self, depth, num_channels, groups, bot_mul):
    net = nn.Sequential()
    for i in range(depth):
        if i == 0:
            net.add(d2l.ResNeXtBlock(
                num_channels, groups, bot_mul, use_1x1conv=True, strides=2))
        else:
            net.add(d2l.ResNeXtBlock(
                num_channels, num_channels, groups, bot_mul))
    return net


In [ ]:
%%tab pytorch
@d2l.add_to_class(AnyNet)
def stage(self, depth, num_channels, groups, bot_mul):
    blk = []
    for i in range(depth):
        if i == 0:
            blk.append(d2l.ResNeXtBlock(num_channels, groups, bot_mul,
                use_1x1conv=True, strides=2))
        else:
            blk.append(d2l.ResNeXtBlock(num_channels, groups, bot_mul))
    return nn.Sequential(*blk)


In [ ]:
%%tab tensorflow
@d2l.add_to_class(AnyNet)
def stage(self, depth, num_channels, groups, bot_mul):
    net = tf.keras.models.Sequential()
    for i in range(depth):
        if i == 0:
            net.add(d2l.ResNeXtBlock(num_channels, groups, bot_mul,
                use_1x1conv=True, strides=2))
        else:
            net.add(d2l.ResNeXtBlock(num_channels, groups, bot_mul))
    return net


In [ ]:
%%tab jax
@d2l.add_to_class(AnyNet)
def stage(self, depth, num_channels, groups, bot_mul):
    blk = []
    for i in range(depth):
        if i == 0:
            blk.append(d2l.ResNeXtBlock(num_channels, groups, bot_mul,
                use_1x1conv=True, strides=(2, 2), training=self.training))
        else:
            blk.append(d2l.ResNeXtBlock(num_channels, groups, bot_mul,
                                        training=self.training))
    return nn.Sequential(blk)




En assemblant la tige, le corps et la tête du réseau, nous complétons l'implémentation d'AnyNet.



In [ ]:
%%tab pytorch, mxnet, tensorflow
@d2l.add_to_class(AnyNet)
def __init__(self, arch, stem_channels, lr=0.1, num_classes=10):
    super(AnyNet, self).__init__()
    self.save_hyperparameters()
    if tab.selected('mxnet'):
        self.net = nn.Sequential()
        self.net.add(self.stem(stem_channels))
        for i, s in enumerate(arch):
            self.net.add(self.stage(*s))
        self.net.add(nn.GlobalAvgPool2D(), nn.Dense(num_classes))
        self.net.initialize(init.Xavier())
    if tab.selected('pytorch'):
        self.net = nn.Sequential(self.stem(stem_channels))
        for i, s in enumerate(arch):
            self.net.add_module(f'stage{i+1}', self.stage(*s))
        self.net.add_module('head', nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
            nn.LazyLinear(num_classes)))
        self.net.apply(d2l.init_cnn)
    if tab.selected('tensorflow'):
        self.net = tf.keras.models.Sequential(self.stem(stem_channels))
        for i, s in enumerate(arch):
            self.net.add(self.stage(*s))
        self.net.add(tf.keras.models.Sequential([
            tf.keras.layers.GlobalAvgPool2D(),
            tf.keras.layers.Dense(units=num_classes)]))


In [ ]:
%%tab jax
@d2l.add_to_class(AnyNet)
def create_net(self):
    net = nn.Sequential([self.stem(self.stem_channels)])
    for i, s in enumerate(self.arch):
        net.layers.extend([self.stage(*s)])
    net.layers.extend([nn.Sequential([
        lambda x: nn.avg_pool(x, window_shape=x.shape[1:3],
                            strides=x.shape[1:3], padding='valid'),
        lambda x: x.reshape((x.shape[0], -1)),
        nn.Dense(self.num_classes)])])
    return net




## Distributions et paramètres des espaces de conception

Comme nous venons d'en discuter dans la :numref:`subsec_the-anynet-design-space`, les paramètres d'un espace de conception sont des hyperparamètres des réseaux dans cet espace de conception.
Considérons le problème de l'identification de bons paramètres dans l'espace de conception AnyNet. Nous pourrions essayer de trouver le *meilleur choix unique* de paramètres pour une quantité donnée de calcul (par exemple, les FLOPs et le temps de calcul). Si nous n'autorisions même que *deux* choix possibles pour chaque paramètre, nous devrions explorer $2^{17} = 131072$ combinaisons pour trouver la meilleure solution. C'est manifestement irréalisable en raison de son coût exorbitant. Pire encore, nous n'apprenons rien de cet exercice sur la manière dont on devrait concevoir un réseau. La prochaine fois que nous ajouterons, par exemple, une étape X, ou une opération de décalage, ou autre, nous devrions repartir de zéro. Pire encore, en raison de la stochasticité de l'entraînement (arrondis, mélange, erreurs de bits), il est peu probable que deux exécutions produisent exactement les mêmes résultats. Une meilleure stratégie consisterait à essayer de déterminer des lignes directrices générales sur la manière dont les choix de paramètres devraient être liés. Par exemple, le rapport de goulot d'étranglement, le nombre de canaux, de blocs, de groupes ou leur évolution entre les couches devraient idéalement être régis par une collection de règles simples. L'approche de :citet:`radosavovic2019network` repose sur les quatre hypothèses suivantes :

1. Nous supposons que des principes de conception généraux existent réellement, de sorte que de nombreux réseaux satisfaisant à ces exigences devraient offrir de bonnes performances. Par conséquent, identifier une *distribution* sur les réseaux peut être une stratégie judicieuse. En d'autres termes, nous supposons qu'il y a beaucoup de bonnes aiguilles dans la botte de foin.
1. Il n'est pas nécessaire d'entraîner les réseaux jusqu'à convergence avant de pouvoir évaluer si un réseau est bon. Au lieu de cela, il suffit d'utiliser les résultats intermédiaires comme guide fiable pour la précision finale. L'utilisation de substituts (approximatifs) pour optimiser un objectif est appelée optimisation multi-fidélité :cite:`forrester2007multi`. Par conséquent, l'optimisation de la conception est effectuée sur la base de la précision obtenue après seulement quelques passages sur le jeu de données, ce qui réduit considérablement le coût. 
1. Les résultats obtenus à plus petite échelle (pour des réseaux plus petits) se généralisent aux plus grands. Par conséquent, l'optimisation est effectuée pour des réseaux structurellement similaires, mais avec un plus petit nombre de blocs, moins de canaux, etc. Ce n'est qu'à la fin que nous devrons vérifier que les réseaux ainsi trouvés offrent également de bonnes performances à grande échelle. 
1. Les aspects de la conception peuvent être approximativement factorisés de sorte qu'il soit possible d'inférer leur effet sur la qualité du résultat de manière quelque peu indépendante. En d'autres termes, le problème d'optimisation est modérément facile.

Ces hypothèses nous permettent de tester de nombreux réseaux à peu de frais. En particulier, nous pouvons *échantillonner* uniformément l'espace des configurations et évaluer leurs performances. Par la suite, nous pouvons évaluer la qualité du choix des paramètres en examinant la *distribution* de l'erreur/précision qui peut être obtenue avec lesdits réseaux. Notons $F(e)$ la fonction de répartition (CDF) pour les erreurs commises par les réseaux d'un espace de conception donné, tirés selon la distribution de probabilité $p$. C'est-à-dire :

$$F(e, p) \stackrel{\textrm{def}}{=} P_{\textrm{net} \sim p} \{e(\textrm{net}) \leq e\}.$$

Notre objectif est maintenant de trouver une distribution $p$ sur les *réseaux* telle que la plupart des réseaux aient un taux d'erreur très bas et où le support de $p$ soit concis. Bien entendu, cela est impossible à réaliser avec précision d'un point de vue informatique. Nous recourons à un échantillon de réseaux $\mathcal{Z} \stackrel{\textrm{def}}{=} \{\textrm{net}_1, \ldots \textrm{net}_n\}$ (avec les erreurs respectives $e_1, \ldots, e_n$) provenant de $p$ et utilisons à la place la CDF empirique $\hat{F}(e, \mathcal{Z})$ :

$$\hat{F}(e, \mathcal{Z}) = \frac{1}{n}\sum_{i=1}^n \mathbf{1}(e_i \leq e).$$

Chaque fois que la CDF pour un ensemble de choix majore (ou égale) une autre CDF, il s'ensuit que son choix de paramètres est supérieur (ou indifférent). En conséquence, :citet:`Radosavovic.Kosaraju.Girshick.ea.2020` ont expérimenté avec un rapport de goulot d'étranglement partagé $k_i = k$ pour toutes les étapes $i$ du réseau. Cela élimine trois des quatre paramètres régissant le rapport de goulot d'étranglement. Pour évaluer si cela affecte (négativement) la performance, on peut tirer des réseaux des distributions contraintes et non contraintes et comparer les CDF correspondantes. Il s'avère que cette contrainte n'affecte pas du tout la précision de la distribution des réseaux, comme on peut le voir dans le premier panneau de la :numref:`fig_regnet-fig`. 
De même, nous pourrions choisir de prendre la même largeur de groupe $g_i = g$ présente aux différentes étapes du réseau. Là encore, cela n'affecte pas les performances, comme on peut le voir dans le deuxième panneau de la :numref:`fig_regnet-fig`.
Ces deux étapes combinées réduisent le nombre de paramètres libres de six. 

![Comparaison des fonctions de répartition empiriques des erreurs des espaces de conception. $\textrm{AnyNet}_\mathit{A}$ est l'espace de conception original ; $\textrm{AnyNet}_\mathit{B}$ lie les rapports de goulot d'étranglement, $\textrm{AnyNet}_\mathit{C}$ lie également les largeurs de groupe, $\textrm{AnyNet}_\mathit{D}$ augmente la profondeur du réseau à travers les étapes. De gauche à droite : (i) lier les rapports de goulot d'étranglement n'a aucun effet sur les performances ; (ii) lier les largeurs de groupe n'a aucun effet sur les performances ; (iii) augmenter les largeurs de réseau (canaux) à travers les étapes améliore les performances ; (iv) augmenter les profondeurs de réseau à travers les étapes améliore les performances. Figure avec l'aimable autorisation de :citet:`Radosavovic.Kosaraju.Girshick.ea.2020`.](../img/regnet-fig.png)
:label:`fig_regnet-fig`

Ensuite, nous cherchons des moyens de réduire la multitude de choix potentiels pour la largeur et la profondeur des étapes. Il est raisonnable de supposer que, à mesure que nous allons plus en profondeur, le nombre de canaux devrait augmenter, c'est-à-dire $c_i \geq c_{i-1}$ ($w_{i+1} \geq w_i$ selon leur notation dans la :numref:`fig_regnet-fig`), donnant $\textrm{AnyNetX}_D$. De même, il est tout aussi raisonnable de supposer qu'au fur et à mesure que les étapes progressent, elles devraient devenir plus profondes, c'est-à-dire $d_i \geq d_{i-1}$, donnant $\textrm{AnyNetX}_E$. Cela peut être vérifié expérimentalement dans les troisième et quatrième panneaux de la :numref:`fig_regnet-fig`, respectivement.

## RegNet

L'espace de conception $\textrm{AnyNetX}_E$ qui en résulte se compose de réseaux simples suivant des principes de conception faciles à interpréter :

* Partager le rapport de goulot d'étranglement $k_i = k$ pour toutes les étapes $i$ ;
* Partager la largeur de groupe $g_i = g$ pour toutes les étapes $i$ ;
* Augmenter la largeur du réseau à travers les étapes : $c_{i} \leq c_{i+1}$ ;
* Augmenter la profondeur du réseau à travers les étapes : $d_{i} \leq d_{i+1}$.

Cela nous laisse avec un dernier ensemble de choix : comment choisir les valeurs spécifiques pour les paramètres ci-dessus de l'espace de conception $\textrm{AnyNetX}_E$ final. En étudiant les réseaux les plus performants de la distribution dans $\textrm{AnyNetX}_E$, on peut observer ce qui suit : la largeur du réseau augmente idéalement de manière linéaire avec l'indice de bloc à travers le réseau, c'est-à-dire $c_j \approx c_0 + c_a j$, où $j$ est l'indice du bloc et la pente $c_a > 0$. Étant donné que nous ne pouvons choisir une largeur de bloc différente que par étape, nous arrivons à une fonction constante par morceaux, conçue pour correspondre à cette dépendance. De plus, les expériences montrent également qu'un rapport de goulot d'étranglement de $k = 1$ donne les meilleurs résultats, c'est-à-dire qu'il est conseillé de ne pas utiliser de goulots d'étranglement du tout. 

Nous recommandons au lecteur intéressé de consulter plus de détails sur la conception de réseaux spécifiques pour différentes quantités de calcul en parcourant :citet:`Radosavovic.Kosaraju.Girshick.ea.2020`. Par exemple, une variante RegNetX efficace à 32 couches est donnée par $k = 1$ (pas de goulot d'étranglement), $g = 16$ (la largeur de groupe est de 16), $c_1 = 32$ et $c_2 = 80$ canaux pour les première et deuxième étapes respectivement, choisies pour avoir une profondeur de $d_1=4$ et $d_2=6$ blocs. L'idée étonnante de cette conception est qu'elle s'applique toujours, même lors de l'étude de réseaux à plus grande échelle. Mieux encore, elle est même valable pour les conceptions de réseaux Squeeze-and-Excitation (SE) (RegNetY) qui possèdent une activation globale des canaux :cite:`Hu.Shen.Sun.2018`.



In [ ]:
%%tab pytorch, mxnet, tensorflow
class RegNetX32(AnyNet):
    def __init__(self, lr=0.1, num_classes=10):
        stem_channels, groups, bot_mul = 32, 16, 1
        depths, channels = (4, 6), (32, 80)
        super().__init__(
            ((depths[0], channels[0], groups, bot_mul),
             (depths[1], channels[1], groups, bot_mul)),
            stem_channels, lr, num_classes)


In [ ]:
%%tab jax
class RegNetX32(AnyNet):
    lr: float = 0.1
    num_classes: int = 10
    stem_channels: int = 32
    arch: tuple = ((4, 32, 16, 1), (6, 80, 16, 1))




Nous pouvons voir que chaque étape RegNetX réduit progressivement la résolution et augmente les canaux de sortie.



In [ ]:
%%tab mxnet, pytorch
RegNetX32().layer_summary((1, 1, 96, 96))


In [ ]:
%%tab tensorflow
RegNetX32().layer_summary((1, 96, 96, 1))


In [ ]:
%%tab jax
RegNetX32(training=False).layer_summary((1, 96, 96, 1))




## Entraînement

L'entraînement du RegNetX à 32 couches sur le jeu de données Fashion-MNIST se fait exactement comme précédemment.



In [ ]:
%%tab mxnet, pytorch, jax
model = RegNetX32(lr=0.05)
trainer = d2l.Trainer(max_epochs=10, num_gpus=1)
data = d2l.FashionMNIST(batch_size=128, resize=(96, 96))
trainer.fit(model, data)


In [ ]:
%%tab tensorflow
trainer = d2l.Trainer(max_epochs=10)
data = d2l.FashionMNIST(batch_size=128, resize=(96, 96))
with d2l.try_gpu():
    model = RegNetX32(lr=0.01)
    trainer.fit(model, data)




## Discussion

Avec des biais inductifs souhaitables (hypothèses ou préférences) comme la localité et l'invariance par translation (:numref:`sec_why-conv`) pour la vision, les CNN ont été les architectures dominantes dans ce domaine. Cela est resté le cas de LeNet jusqu'à ce que les Transformers (:numref:`sec_transformer`) :cite:`Dosovitskiy.Beyer.Kolesnikov.ea.2021,touvron2021training` commencent à surpasser les CNN en termes de précision. Bien qu'une grande partie des progrès récents concernant les Transformers pour la vision *puisse* être rétroportée dans les CNN :cite:`liu2022convnet`, cela n'est possible qu'à un coût de calcul plus élevé. Tout aussi important, les récentes optimisations matérielles (NVIDIA Ampere et Hopper) n'ont fait qu'élargir l'écart en faveur des Transformers. 

Il est à noter que les Transformers ont un degré de biais inductif vers la localité et l'invariance par translation nettement inférieur à celui des CNN. Si les structures apprises ont prévalu, c'est notamment grâce à la disponibilité de vastes collections d'images, telles que LAION-400m et LAION-5B :cite:`schuhmann2022laion`, comptant jusqu'à 5 milliards d'images. De manière assez surprenante, certains des travaux les plus pertinents dans ce contexte incluent même des MLP :cite:`tolstikhin2021mlp`. 

En résumé, les Transformers pour la vision (:numref:`sec_vision-transformer`) sont désormais en tête en termes de performances de pointe dans la classification d'images à grande échelle, montrant que *la scalabilité l'emporte sur les biais inductifs* :cite:`Dosovitskiy.Beyer.Kolesnikov.ea.2021`. Cela inclut le pré-entraînement de Transformers à grande échelle (:numref:`sec_large-pretraining-transformers`) avec l'auto-attention multi-têtes (:numref:`sec_multihead-attention`). Nous invitons les lecteurs à se plonger dans ces chapitres pour une discussion beaucoup plus détaillée.

## Exercices

1. Augmentez le nombre d'étapes à quatre. Pouvez-vous concevoir un RegNetX plus profond qui soit plus performant ?
1. De-ResNeXt-ifiez les RegNets en remplaçant le bloc ResNeXt par le bloc ResNet. Quelles sont les performances de votre nouveau modèle ?
1. Implémentez plusieurs instances d'une famille « VioNet » en *enfreignant* les principes de conception de RegNetX. Quelles sont leurs performances ? Lequel des facteurs ($d_i$, $c_i$, $g_i$, $b_i$) est le plus important ?
1. Votre objectif est de concevoir le MLP « parfait ». Pouvez-vous utiliser les principes de conception présentés ci-dessus pour trouver de bonnes architectures ? Est-il possible d'extrapoler des petits réseaux aux grands ?

:begin_tab:`mxnet`
[Discussions](https://discuss.d2l.ai/t/7462)
:end_tab:

:begin_tab:`pytorch`
[Discussions](https://discuss.d2l.ai/t/7463)
:end_tab:

:begin_tab:`tensorflow`
[Discussions](https://discuss.d2l.ai/t/8738)
:end_tab:

:begin_tab:`jax`
[Discussions](https://discuss.d2l.ai/t/18009)
:end_tab:
